# Phân tích riêng R, F_event_months, F_events và M — Santander
Acquisition là chuyển 0→1 giữa hai snapshot liền tháng, gồm mở lại. R là số tháng từ event gần nhất; F_event_months đếm tháng có event; F_events đếm từng sản phẩm phát sinh. M_n_products là số sản phẩm đang sở hữu, không phải tiền.

Đánh giá 5 cặp tháng gần nhất. Biểu đồ dùng trung bình đều tỷ lệ các tháng có nhóm; bảng ghi từng tháng chốt → tháng kết quả và mẫu số. Điểm M có trọng số dùng bucket đều 5 điểm, điểm 0 riêng. Không phân tích kết hợp RF, FM hay RFM. Outputs cũ được xóa để tránh nhầm kết quả.


## Nội dung
1. Ingest và kiểm chứng acquisition
2. Tính bốn biến tại tháng chốt
3. Tỷ lệ acquisition từng tháng và trung bình 5 lần đánh giá
4. M có trọng số: bucket cách đều 5 điểm
5. Xuất bảng và giới hạn


In [ ]:
import os, glob, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})

WORK_DIR = os.environ.get('WORK_DIR', '/kaggle/working/')
ZIP_PATH = os.environ.get('SANTANDER_ZIP')
if not ZIP_PATH:
    matches = glob.glob('/kaggle/input/**/train_ver2.csv.zip', recursive=True)
    assert len(matches) == 1, f'Cần đúng 1 ZIP nguồn, thấy: {matches}'
    ZIP_PATH = matches[0]

# Tháng chốt để tính RFM. None = tháng cuối của dữ liệu.
# Mọi R/F/M chỉ dùng dữ liệu có record_date <= AS_OF_MONTH.
AS_OF_MONTH = None

# Cohort onboarding hàng loạt phát hiện ở bước EDA trước (chỉ gắn cờ, không xóa)
ONBOARDING_MONTH = pd.Timestamp('2015-07-28')
ONBOARDING_IDX = ONBOARDING_MONTH.year * 12 + ONBOARDING_MONTH.month

# Các mốc phân vị dùng khi mô tả phân phối F, R (median, P95, P99 để nhìn outlier)
PCTS = [.25, .5, .75, .95, .99]
PCT_LABELS = {'50%': 'median', '95%': 'P95', '99%': 'P99'}

PRODUCT_MAP = {
    'ind_ahor_fin_ult1': 'prod_saving_account', 'ind_aval_fin_ult1': 'prod_guarantees',
    'ind_cco_fin_ult1': 'prod_current_account', 'ind_cder_fin_ult1': 'prod_derivada_account',
    'ind_cno_fin_ult1': 'prod_payroll_account', 'ind_ctju_fin_ult1': 'prod_junior_account',
    'ind_ctma_fin_ult1': 'prod_mas_particular_account', 'ind_ctop_fin_ult1': 'prod_particular_account',
    'ind_ctpp_fin_ult1': 'prod_particular_plus_account', 'ind_deco_fin_ult1': 'prod_short_term_deposit',
    'ind_deme_fin_ult1': 'prod_medium_term_deposit', 'ind_dela_fin_ult1': 'prod_long_term_deposit',
    'ind_ecue_fin_ult1': 'prod_eaccount', 'ind_fond_fin_ult1': 'prod_funds',
    'ind_hip_fin_ult1': 'prod_mortgage', 'ind_plan_fin_ult1': 'prod_pension_plan',
    'ind_pres_fin_ult1': 'prod_loans', 'ind_reca_fin_ult1': 'prod_taxes',
    'ind_tjcr_fin_ult1': 'prod_credit_card', 'ind_valo_fin_ult1': 'prod_securities',
    'ind_viv_fin_ult1': 'prod_home_account', 'ind_nomina_ult1': 'prod_payroll',
    'ind_nom_pens_ult1': 'prod_pension_payroll', 'ind_recibo_ult1': 'prod_direct_debit',
}
PRODUCT_COLS_RAW = list(PRODUCT_MAP)
PRODUCT_COLS = list(PRODUCT_MAP.values())
KEY_COLS = ['customer_id', 'record_date']
FAMILY = ['prod_payroll', 'prod_pension_payroll']   # hai cờ gần như trùng nhau (xem 4.2)
assert len(PRODUCT_COLS) == 24
rename_dict = {'fecha_dato': 'record_date', 'ncodpers': 'customer_id',
               'renta': 'household_income', **PRODUCT_MAP}
RAW_NEEDED_COLS = ['fecha_dato', 'ncodpers', 'renta'] + PRODUCT_COLS_RAW
# Quy ước chung cho toàn notebook: màu theo ý nghĩa, không theo thứ hạng.
from matplotlib.ticker import FuncFormatter, MaxNLocator
CHART_BLUE = '#3675A9'
CHART_GRAY = '#89939D'
CHART_ORANGE = '#D88935'
CHART_BASELINE = '#B85D35'
MONTH_COLORS = ['#3675A9', '#429E83', '#8B65AA']
plt.rcParams.update({'font.size': 10, 'axes.titlesize': 12,
                     'axes.labelsize': 10, 'figure.facecolor': 'white',
                     'axes.facecolor': 'white', 'axes.axisbelow': True})

def rate_axis(ax, upper):
    ax.set_ylim(0, upper)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}%'))
    ax.grid(False, axis='x')
    ax.grid(True, axis='y', alpha=.22)
    ax.set_ylabel('Khách có acquisition tháng kế tiếp (%)')

def month_pair(m):
    return f'Chốt {m} → kết quả {(pd.Period(m, freq="M") + 1).strftime("%Y-%m")}'

# Bin cố định: không thay đổi theo quantile hoặc theo tháng.
SCORE_EDGES = [0, 1, 3, 5, 10, 20, np.inf]
SCORE_LABELS = ['0 điểm', '>0–1', '>1–3', '>3–5', '>5–10', '>10–20', '>20']

def fixed_score_bucket(points):
    points = pd.Series(points)
    out = pd.Series(pd.NA, index=points.index, dtype='object')
    out.loc[points.eq(0)] = SCORE_LABELS[0]
    positive = points.gt(0)
    out.loc[positive] = pd.cut(points.loc[positive], SCORE_EDGES,
                              labels=SCORE_LABELS[1:]).astype(object)
    return pd.Categorical(out, categories=SCORE_LABELS, ordered=True)


---
## 1. Ingest và kiểm tra hợp đồng dữ liệu

`assert` khóa `(customer_id, record_date)` duy nhất, giá trị sản phẩm chỉ 0/1/NaN; kiểm tra tập tháng liên tục. NaN được giữ nguyên, không fill. Dữ liệu được sắp theo `(customer_id, record_date)`: các bước sau (event) dựa vào thứ tự dòng này.

In [ ]:
dtype_map = {c: 'float32' for c in PRODUCT_COLS_RAW}
dtype_map.update({'ncodpers': 'int32', 'renta': 'float32'})
with zipfile.ZipFile(ZIP_PATH) as zf:
    with zf.open('train_ver2.csv') as source:
        df = pd.read_csv(source, usecols=RAW_NEEDED_COLS, dtype=dtype_map,
                         na_values=['NA', ' NA'], parse_dates=['fecha_dato'])
df = df.rename(columns=rename_dict)
df['month_idx'] = (df.record_date.dt.year * 12 + df.record_date.dt.month).astype('int32')
df = df.sort_values(['customer_id', 'record_date']).reset_index(drop=True)

assert df.customer_id.notna().all() and df.record_date.notna().all()
n_dup = df.duplicated(['customer_id', 'record_date']).sum()
assert n_dup == 0, f'{n_dup:,} khóa khách-tháng trùng'
n_bad = (~df[PRODUCT_COLS].isin([0, 1]) & df[PRODUCT_COLS].notna()).sum().sum()
assert n_bad == 0, f'{n_bad:,} giá trị sản phẩm ngoài 0/1/NaN'

FIRST_MONTH, LAST_MONTH = df.record_date.min(), df.record_date.max()
LAST_IDX = LAST_MONTH.year * 12 + LAST_MONTH.month
AS_OF = pd.Timestamp(AS_OF_MONTH) if AS_OF_MONTH else LAST_MONTH
assert AS_OF in set(df.record_date.unique()), 'AS_OF phải là một tháng có trong dữ liệu'
AS_OF_IDX = AS_OF.year * 12 + AS_OF.month

print(f'{len(df):,} dòng | {df.customer_id.nunique():,} khách | {df.memory_usage(deep=True).sum()/1e9:.2f} GB')
print(f'Phạm vi: {FIRST_MONTH.date()} – {LAST_MONTH.date()} | as_of = {AS_OF.date()}')

months = np.sort(df.month_idx.unique())
print(f'Số tháng: {len(months)} | liên tục: {bool((np.diff(months) == 1).all())}')
assert (np.diff(months) == 1).all(), 'Các tháng nguồn phải liên tục'


### 1.1. Cửa sổ quan sát của từng khách

**Kiểm tra gì → quyết định gì:** khách có bao nhiêu tháng, có tháng hụt không (tháng hụt làm mất cơ hội bắt event) và ai thuộc cohort 07/2015 (cần cờ riêng khi đọc F). `first_idx` là ranh giới quan sát trong dataset, không phải ngày mở tài khoản, nên tenure bị cắt trái.

In [ ]:
window = (df.groupby('customer_id', sort=False)
            .agg(first_idx=('month_idx', 'min'), last_idx=('month_idx', 'max'),
                 n_months=('month_idx', 'count')).reset_index())
window['span_months'] = window.last_idx - window.first_idx + 1
window['missing_months_in_span'] = window.span_months - window.n_months
window['is_onboarding_batch'] = window.first_idx.eq(ONBOARDING_IDX)
COHORT = window.set_index('customer_id').is_onboarding_batch

print(f"Chỉ 1 tháng: {(window.n_months == 1).mean():.1%} | "
      f"có tháng hụt: {(window.missing_months_in_span > 0).mean():.1%} | "
      f"vào cohort 07/2015: {window.is_onboarding_batch.mean():.1%}")

## 2. Chất lượng dữ liệu
Không điền 0 vào cờ sản phẩm thiếu; không nối event qua tháng hụt. Cờ cohort 07/2015 biểu thị tháng xuất hiện đầu tiên trong dữ liệu, chưa chứng minh khách mới thực sự.

In [ ]:
cols = KEY_COLS + PRODUCT_COLS + ['household_income']
miss = df[cols].isna().sum()
assert miss[KEY_COLS].sum() == 0, 'Khóa bị NaN: dừng lại'
raw_tbl = pd.DataFrame({'n_missing': miss, 'pct_missing': (miss / len(df) * 100).round(3)})
print(f'{int((miss == 0).sum())}/{len(miss)} cột không thiếu. Các cột có thiếu:')
display(raw_tbl[raw_tbl.n_missing > 0].sort_values('n_missing', ascending=False))

PROD_NAN_COLS = [c for c in PRODUCT_COLS if df[c].isna().any()]
print('Cột sản phẩm có NaN:', PROD_NAN_COLS or '(không có)')
if PROD_NAN_COLS:
    nan_months = df.loc[df[PROD_NAN_COLS].isna().any(axis=1), 'record_date'].dt.strftime('%Y-%m').unique()
    print('Tháng có NaN sản phẩm:', ', '.join(sorted(nan_months)))

g = df.groupby('customer_id', sort=False)
consecutive = (df.month_idx - g.month_idx.shift()).eq(1)
n_pairs = int(consecutive.sum())
if PROD_NAN_COLS:
    rows = []
    for c in PROD_NAN_COLS:
        undetermined = consecutive & (g[c].shift().isna() | df[c].isna())
        rows.append((c, int(undetermined.sum()), undetermined.sum() / n_pairs * 100))
    display(pd.DataFrame(rows, columns=['cot', 'cap_khong_xac_dinh', 'pct_tren_tong_cap']).round(4))
    any_nan_customers = df.loc[df[PROD_NAN_COLS].isna().any(axis=1), 'customer_id'].nunique()
    print(f'Tổng cặp tháng liền kề: {n_pairs:,} | khách có ≥1 NaN sản phẩm: {any_nan_customers:,} '
          f'({any_nan_customers / df.customer_id.nunique():.2%})')
else:
    print('Không có NaN ở cột sản phẩm: không mất event nào.')
del g, consecutive

## 3. Acquisition events và kiểm tra
Chỉ tính 0→1 trên hai tháng liên tiếp; snapshot đầu tiên không tạo event. Giữ kiểm tra độc lập để bảo vệ định nghĩa này.

In [ ]:
def build_acquisition_events(snapshots, product_cols):
    s = snapshots.sort_values(['customer_id', 'record_date']).reset_index(drop=True)
    g = s.groupby('customer_id', sort=False)
    gap = s['month_idx'] - g['month_idx'].shift()
    prev = g[product_cols].shift()
    consecutive = gap.eq(1).to_numpy()[:, None]
    mask = (prev.eq(0) & s[product_cols].eq(1)).to_numpy() & consecutive
    rows, cols = np.nonzero(mask)
    out = s.iloc[rows][['customer_id', 'record_date', 'month_idx']].copy()
    out['row'] = rows                                   # vị trí dòng trong snapshots đã sắp
    out['product'] = np.asarray(product_cols, dtype=object)[cols]
    return out.sort_values(['customer_id', 'record_date', 'product']).reset_index(drop=True)


def make_toy(rows, cols):
    t = pd.DataFrame(rows, columns=['customer_id', 'month_idx'] + cols)
    t['record_date'] = pd.to_datetime([f'{(m - 1) // 12}-{(m - 1) % 12 + 1:02d}-28' for m in t.month_idx])
    return t


# Unit test: 2 sản phẩm cùng tháng, NaN, tháng hụt, 1→0→1, snapshot đầu đã có sản phẩm
toy = pd.DataFrame([
    (1, '2015-01-28', 0, 0, np.nan),
    (1, '2015-02-28', 1, 1, 1),        # A,B: 0→1 (C: prev NaN → bỏ)
    (1, '2015-04-28', 0, 1, 1),        # hụt tháng 3 → không tính
    (1, '2015-05-28', 1, 0, 1),        # A: 0→1
    (2, '2015-01-28', 1, 0, 0),        # A có sẵn ở snapshot đầu → không phải event
    (2, '2015-02-28', 0, 0, 0),
    (2, '2015-03-28', 1, 0, 0),        # A: 0→1 (mua lại)
], columns=['customer_id', 'record_date', 'A', 'B', 'C'])
toy['record_date'] = pd.to_datetime(toy.record_date)
toy['month_idx'] = toy.record_date.dt.year * 12 + toy.record_date.dt.month
toy_ev = build_acquisition_events(toy, ['A', 'B', 'C'])
got = set(toy_ev[['customer_id', 'record_date', 'product']].itertuples(index=False, name=None))
assert got == {(1, pd.Timestamp('2015-02-28'), 'A'), (1, pd.Timestamp('2015-02-28'), 'B'),
               (1, pd.Timestamp('2015-05-28'), 'A'), (2, pd.Timestamp('2015-03-28'), 'A')}
print('PASS: test event (NaN, tháng hụt, nhiều sản phẩm cùng tháng, snapshot đầu)')

In [ ]:
events = build_acquisition_events(df, PRODUCT_COLS)
# `row` phải trỏ đúng dòng của df (df đã sắp cùng thứ tự với hàm build)
assert (df.customer_id.to_numpy()[events.row] == events.customer_id.to_numpy()).all()
assert (df.month_idx.to_numpy()[events.row] == events.month_idx.to_numpy()).all()
events['is_onboarding_batch'] = events.customer_id.map(COHORT).astype(bool)
print(f'{len(events):,} acquisition event | {events.customer_id.nunique():,} khách có ≥1 event')

In [ ]:
def count_transitions_full(snapshots, product_cols, before, after):
    counts = pd.Series(0, index=product_cols, dtype='int64')
    by_month = {m: g.set_index('customer_id')[product_cols]
                for m, g in snapshots.groupby('month_idx')}
    ms = sorted(by_month)
    for p, c in zip(ms[:-1], ms[1:]):
        if c - p != 1:
            continue
        common = by_month[p].index.intersection(by_month[c].index)
        counts += (by_month[p].loc[common].eq(before) & by_month[c].loc[common].eq(after)).sum()
    return counts

full = count_transitions_full(df, PRODUCT_COLS, 0, 1)
assert len(events) == int(full.sum()), (len(events), int(full.sum()))
assert not events.duplicated(['customer_id', 'record_date', 'product']).any()
assert events.record_date.gt(FIRST_MONTH).all()
print(f'PASS: đếm độc lập khớp {len(events):,} event')

## Frequency: phân tích cả hai định nghĩa
F_events đếm từng event sản phẩm (mở nhiều sản phẩm trong một tháng được đếm nhiều lần). F_event_months đếm số tháng có ít nhất một event. Cả hai đều tính từ đầu lịch sử quan sát đến tháng chốt, gồm mở lại.


In [ ]:
f_events = events.groupby('customer_id').size()
f_months = events.groupby('customer_id').month_idx.nunique()
cmp = pd.DataFrame({'F_events': f_events, 'F_event_months': f_months})
print(f"Khách có F_events > F_event_months (mua nhiều SP cùng tháng): {(cmp.F_events > cmp.F_event_months).mean():.1%}")
print(cmp.describe(percentiles=PCTS).round(2).rename(index=PCT_LABELS))

## Dựng biến tại tháng chốt
Chỉ dùng lịch sử đến tháng chốt. F_events là số event; F_event_months là số tháng có event. Cột F giữ làm alias của F_events.


In [ ]:
def build_rfm(snapshots, events, product_cols, as_of_idx):
    snap = snapshots[snapshots.month_idx.le(as_of_idx)]
    now = snap[snap.month_idx.eq(as_of_idx)].copy()
    now['M_n_products'] = now[product_cols].sum(axis=1, min_count=len(product_cols))
    base = now[['customer_id', 'M_n_products']]

    ev = events[events.month_idx.le(as_of_idx)]
    g = ev.groupby('customer_id')
    ev_feat = pd.DataFrame({'F': g.size(), 'F_event_months': g.month_idx.nunique(),
                            'last_event_idx': g.month_idx.max()})

    inc = snap.groupby('customer_id').agg(first_idx=('month_idx', 'min'),
                                          M_renta=('household_income', 'last'))
    out = (base.merge(ev_feat, left_on='customer_id', right_index=True, how='left')
               .merge(inc, left_on='customer_id', right_index=True, how='left'))
    for c in ['F', 'F_event_months']:
        out[c] = out[c].fillna(0).astype('int32')
    out['F_events'] = out['F']  # F giữ lại để tương thích output cũ
    out['has_event'] = out.F_events.gt(0)
    out['R'] = (as_of_idx - out.last_event_idx)                # NaN nếu chưa có event
    out['tenure_months'] = (as_of_idx - out.first_idx).astype('int32')
    out['renta_missing'] = out.M_renta.isna()
    out['as_of_month'] = pd.Timestamp(year=(as_of_idx - 1) // 12, month=(as_of_idx - 1) % 12 + 1, day=28)
    return out.drop(columns=['last_event_idx', 'first_idx'])


# Test: không rò rỉ tương lai, khách F=0, khách vắng ở as_of
M0 = 2015 * 12
toy2 = make_toy([
    (1, M0 + 1, 0, 0, 100.0), (1, M0 + 2, 1, 0, np.nan), (1, M0 + 3, 1, 1, np.nan), (1, M0 + 4, 1, 1, np.nan),
    (2, M0 + 1, 1, 0, np.nan), (2, M0 + 2, 1, 0, np.nan), (2, M0 + 3, 1, 0, np.nan), (2, M0 + 4, 1, 0, np.nan),
    (3, M0 + 1, 0, 0, 50.0), (3, M0 + 2, 1, 0, 50.0),                                   # vắng ở as_of
    (4, M0 + 1, 1, 0, np.nan), (4, M0 + 2, 0, 0, np.nan), (4, M0 + 3, 1, 0, np.nan), (4, M0 + 4, 1, 0, np.nan),
], ['A', 'B', 'household_income'])
toy2_ev = build_acquisition_events(toy2, ['A', 'B'])
r3 = build_rfm(toy2, toy2_ev, ['A', 'B'], M0 + 3).set_index('customer_id')
assert set(r3.index) == {1, 2, 4}                                # khách 3 vắng ở as_of
assert r3.loc[1, 'F'] == 2 and r3.loc[1, 'R'] == 0 and r3.loc[1, 'M_n_products'] == 2
assert r3.loc[1, 'M_renta'] == 100.0                             # renta hợp lệ gần nhất
assert r3.loc[2, 'F'] == 0 and np.isnan(r3.loc[2, 'R']) and not r3.loc[2, 'has_event']
assert r3.loc[4, 'F'] == 1 and r3.loc[4, 'R'] == 0
r2 = build_rfm(toy2, toy2_ev, ['A', 'B'], M0 + 2).set_index('customer_id')
assert r2.loc[1, 'F'] == 1                                       # event tháng 3 không rò vào as_of=tháng 2
assert r2.loc[4, 'F'] == 0                                       # event tháng 3 của khách 4 chưa tồn tại
r4 = build_rfm(toy2, toy2_ev, ['A', 'B'], M0 + 4).set_index('customer_id')
assert r4.loc[1, 'R'] == 1 and r4.loc[4, 'R'] == 1
print('PASS: test RFM (không rò rỉ tương lai, F=0, khách vắng ở as_of)')

In [ ]:
rfm = build_rfm(df, events, PRODUCT_COLS, AS_OF_IDX)
rfm = rfm.merge(window[['customer_id', 'is_onboarding_batch']], on='customer_id', how='left')
assert rfm.customer_id.is_unique
assert rfm.R.dropna().ge(0).all()
assert rfm.loc[rfm.has_event, 'R'].notna().all() and rfm.loc[~rfm.has_event, 'R'].isna().all()
print(f'{len(rfm):,} khách tại as_of = {AS_OF.date()}')

In [ ]:
rfm_cols = ['F', 'R', 'M_n_products', 'M_renta']
m = rfm[rfm_cols].isna().sum()
display(pd.DataFrame({'n_missing': m, 'pct_missing': (m / len(rfm) * 100).round(3)}))
assert rfm.F.notna().all()
assert rfm.R.isna().eq(~rfm.has_event).all(), 'R thiếu không khớp F=0'
assert (df.household_income.dropna() > 0).all(), 'renta ≤ 0: phải coi là thiếu'

# renta: lấy giá trị hợp lệ gần nhất có cứu thêm khách nào không?
at_asof = df.loc[df.month_idx.eq(AS_OF_IDX)].set_index('customer_id').household_income
print(f"renta thiếu tại snapshot as_of: {rfm.customer_id.map(at_asof).isna().mean():.2%} "
      f"| sau khi lấy giá trị gần nhất: {rfm.renta_missing.mean():.2%}")
n0 = rfm.M_n_products.eq(0)
print(f'n_products = 0: {int(n0.sum()):,} khách ({n0.mean():.1%})')

display(rfm.groupby('renta_missing').agg(
    n_khach=('customer_id', 'size'), pct_F_duong=('has_event', 'mean'), F_tb=('F', 'mean'),
    n_products_tb=('M_n_products', 'mean'), tenure_tb=('tenure_months', 'mean'),
    pct_cohort_0715=('is_onboarding_batch', 'mean')).round(3))

## 6. Phân phối cần thiết và khác biệt cohort
Nhóm chưa quan sát acquisition cần tách riêng. Các bảng mô tả dùng để chọn mức gộp hợp lý, không chứng minh khả năng dự đoán.

In [ ]:
display(
    rfm[['R', 'F_event_months', 'F_events', 'M_n_products']]
    .describe(percentiles=PCTS)
    .round(2)
)

print(f'Chưa thấy acquisition: {(~rfm.has_event).mean():.1%}')

for col in ['R', 'F_event_months', 'F_events', 'M_n_products']:
    values = rfm[col].dropna()

    if col in ['F_event_months', 'F_events']:
        values = values.loc[values.gt(0)]

    counts = values.value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(10, 4.8))
    ax.bar(counts.index, counts.values, color=CHART_BLUE)

    # Chỉ hiện các mức được vẽ, không hiện tick 0 cho F.
    ax.set_xticks(counts.index)
    ax.yaxis.set_major_formatter(
        FuncFormatter(lambda v, _: f'{v:,.0f}')
    )

    if col == 'R':
        note = ' — chỉ khách có event'
    elif col in ['F_event_months', 'F_events']:
        note = ' — chỉ khách có F > 0'
    else:
        note = ''

    ax.set_title(f'{col} | chốt {AS_OF:%Y-%m}{note}')
    ax.set_ylabel('Số khách')
    ax.grid(False, axis='x')

    plt.tight_layout()
    plt.show()

## 7. Khách nào có sản phẩm phát sinh ở tháng sau?

**Ví dụ tháng chốt 04/2016 → xem kết quả 05/2016:**

| R tại tháng chốt | Nghĩa |
|---|---|
| R = 0 | Vừa phát sinh sản phẩm trong tháng 04/2016 |
| R = 1 | Lần gần nhất là tháng 03/2016 |
| R = 2 | Lần gần nhất là tháng 02/2016 |
| Chưa thấy event | Chưa quan sát được acquisition trong lịch sử có sẵn; không phải R = 0 |

Tỷ lệ tháng sau = số khách có ít nhất một sản phẩm chuyển 0→1 trong 05/2016 / số khách đủ dữ liệu trong nhóm.

Chỉ dùng lịch sử đến tháng chốt để tính R/F/số sản phẩm sở hữu. Chỉ đánh giá khách có đủ 24 cờ sản phẩm ở cả tháng chốt và tháng sau. Khách vắng/thiếu được loại khỏi mẫu, không gán thành không phát sinh. Acquisition gồm cả mở lại sản phẩm.

Phần chính trình bày tháng chốt cuối có kết quả tháng sau. Bảng đối chiếu các tháng đặt phía dưới để kiểm tra độ ổn định.

In [ ]:
N_EVAL_MONTHS = 5
MIN_GROUP_N = 100

def make_response_panel(snapshots, acquisition_events, product_cols, cutoff):
    current = snapshots.loc[snapshots.month_idx.eq(cutoff), ['customer_id'] + product_cols]
    following = snapshots.loc[snapshots.month_idx.eq(cutoff+1), ['customer_id'] + product_cols]
    valid_now = current.loc[current[product_cols].notna().all(axis=1), 'customer_id']
    valid_next = following.loc[following[product_cols].notna().all(axis=1), 'customer_id']
    eligible = pd.Index(valid_now).intersection(pd.Index(valid_next))
    features = build_rfm(snapshots, acquisition_events, product_cols, cutoff)
    panel = features.loc[features.customer_id.isin(eligible)].copy()
    responders = acquisition_events.loc[acquisition_events.month_idx.eq(cutoff+1), 'customer_id'].unique()
    panel['target_next_month'] = panel.customer_id.isin(responders).astype('int8')
    panel['is_onboarding_batch'] = panel.customer_id.map(COHORT).fillna(False).astype(bool)
    panel['eval_month'] = panel.as_of_month.dt.strftime('%Y-%m')
    audit = {'cutoff': cutoff, 'n_current': len(current), 'n_eligible': len(panel),
             'n_excluded': len(current)-len(panel),
             'response_rate': panel.target_next_month.mean()}
    return panel, audit

# Mức đơn vị cố định, không chia quantile hoặc gom cụm.
R_TOP = 14
F_TOP = 13
M_TOP = 14

def unit_bins(top):
    assert isinstance(top, int) and top >= 1
    return [-1] + list(range(top)) + [np.inf], [str(i) for i in range(top)] + [f'{top}+']

R_EDGES, R_BINS = unit_bins(R_TOP)
F_EDGES, F_LEVELS = unit_bins(F_TOP)
M_EDGES, M_LEVELS = unit_bins(M_TOP)
R_LEVELS = R_BINS + ['No event']

def add_groups(panel):
    panel = panel.copy()
    r = pd.cut(panel.R, R_EDGES, labels=R_BINS)
    panel['R_group'] = r.cat.add_categories(['No event']).fillna('No event')
    panel['F_group'] = pd.cut(panel.F_event_months, F_EDGES, labels=F_LEVELS)
    panel['M_group'] = pd.cut(panel.M_n_products, M_EDGES, labels=M_LEVELS)
    return panel

def rate_table(panel, group):
    table = panel.groupby(['eval_month',group], observed=True).agg(
        n=('target_next_month','size'), acquired=('target_next_month','sum')).reset_index()
    table['rate'] = table.acquired / table.n
    baseline = panel.groupby('eval_month').target_next_month.mean()
    table['baseline'] = table.eval_month.map(baseline)
    table['lift'] = table.rate / table.baseline.replace(0,np.nan)
    z = 1.96; denom = 1 + z*z/table.n
    center = (table.rate + z*z/(2*table.n))/denom
    half = z*np.sqrt(table.rate*(1-table.rate)/table.n + z*z/(4*table.n**2))/denom
    table['low95'], table['high95'] = center-half, center+half
    table['small_group'] = table.n < MIN_GROUP_N
    return table

eval_cutoffs = [int(m) for m in months if m+1 in set(months) and m>months[0]][-N_EVAL_MONTHS:]
assert len(eval_cutoffs) == N_EVAL_MONTHS, 'Chưa đủ số tháng eval yêu cầu'
panels, audits = [], []
for cutoff in eval_cutoffs:
    panel, audit = make_response_panel(df, events, PRODUCT_COLS, cutoff)
    panels.append(add_groups(panel)); audits.append(audit)
response_panel = pd.concat(panels, ignore_index=True)
assert not response_panel.duplicated(['customer_id','eval_month']).any()
display(pd.DataFrame(audits))
F_EVENTS_LEVELS = [str(i) for i in range(int(response_panel.F_events.max())+1)]
response_panel['F_events_group'] = pd.Categorical(
    response_panel.F_events.astype(int).astype(str), categories=F_EVENTS_LEVELS, ordered=True)
assert len(response_panel.eval_month.unique()) == 5
response_tables = {g: rate_table(response_panel,g) for g in ['R_group','F_group','F_events_group','M_group']}

In [ ]:
LEVELS = {'R_group': R_LEVELS, 'F_group': F_LEVELS, 'F_events_group': F_EVENTS_LEVELS, 'M_group': M_LEVELS}
LABELS = LEVELS.copy()   # Giữ tương thích; nhãn chỉ là giá trị mức.
TITLES = {'R_group': 'R: lần phát sinh sản phẩm gần nhất',
          'F_group': 'F_event_months: số tháng có phát sinh sản phẩm',
          'F_events_group': 'F_events: số event sản phẩm',
          'M_group': 'M-proxy: số sản phẩm đang sở hữu'}
eval_months = sorted(response_panel.eval_month.unique())
latest_month = eval_months[-1]
next_month = (pd.Period(latest_month, freq='M') + 1).strftime('%Y-%m')
n_eval = len(eval_months)
baseline_mean = response_panel.groupby('eval_month').target_next_month.mean().mean()
mean_tables = {}
for group in LEVELS:
    part = response_tables[group].groupby(group, observed=True).agg(
        mean_rate=('rate','mean'), mean_baseline=('baseline','mean'),
        n_months=('eval_month','nunique'), customer_months=('n','sum'),
        acquired_months=('acquired','sum')).reindex(LEVELS[group])
    part['lift'] = part.mean_rate / part.mean_baseline.replace(0,np.nan)
    mean_tables[group] = part
    fig, ax = plt.subplots(figsize=(10,4.8))
    bars = ax.bar(range(len(part)), part.mean_rate*100, color='#3675A9', width=.65)
    ax.bar_label(bars, labels=[f'{v:.2%}' if pd.notna(v) else '' for v in part.mean_rate], padding=5, fontsize=10)
    ax.axhline(baseline_mean*100, color='#B85D35', linestyle='--', label=f'Tỷ lệ chung trung bình: {baseline_mean:.2%}')
    tick_step = max(1, int(np.ceil(len(part)/20)))
    ticks = list(range(0,len(part),tick_step))
    ax.set_xticks(ticks); ax.set_xticklabels([LEVELS[group][i] for i in ticks], fontsize=10)
    ax.set_xlim(-.6,len(part)-.4)
    ax.set_ylabel('Tỷ lệ acquisition trung bình (%)')
    ax.set_title(f'{TITLES[group]}\nTrung bình {n_eval} lần đánh giá', fontsize=13, pad=12)
    highest = max(float(part.mean_rate.max()), float(baseline_mean))
    ax.set_ylim(0,max(highest*100*1.23,1))
    ax.grid(False,axis='x'); ax.legend(loc='upper left',fontsize=9)
    plt.tight_layout(); plt.show()
    compact = part[['customer_months','acquired_months','n_months','mean_rate','lift']].copy()
    for c in ['customer_months','acquired_months','n_months']: compact[c]=compact[c].astype('Int64')
    compact['mean_rate']=compact.mean_rate.map(lambda v:f'{v:.2%}' if pd.notna(v) else '—')
    compact['lift']=compact.lift.round(2)
    compact.columns=['Lượt khách-tháng','Lượt phát sinh','Số tháng có dữ liệu','Tỷ lệ TB','Lift']
    display(compact)


### Đối chiếu các tháng eval
Mỗi cột là tháng chốt → tháng kết quả. Trung bình bỏ qua tháng không có nhóm; số tháng có dữ liệu luôn được ghi. Dấu * chỉ ít khách, chưa phải kiểm định thống kê.


In [ ]:
monthly_views = {}
for group in LEVELS:
    print(TITLES[group])
    table = response_tables[group]
    rates = table.pivot(index=group,columns='eval_month',values='rate').reindex(index=LEVELS[group],columns=eval_months)
    counts = table.pivot(index=group,columns='eval_month',values='n').reindex(index=LEVELS[group],columns=eval_months)
    acquired = table.pivot(index=group,columns='eval_month',values='acquired').reindex(index=LEVELS[group],columns=eval_months)
    view = pd.DataFrame(index=rates.index)
    for m in eval_months:
        label = f'{m} → {(pd.Period(m,freq="M")+1).strftime("%Y-%m")}'
        view[label] = [f'{r:.2%} ({int(a):,}/{int(n):,})'+(' *' if n<MIN_GROUP_N else '') if pd.notna(r) else '—' for r,a,n in zip(rates[m],acquired[m],counts[m])]
    view['TB tỷ lệ'] = rates.mean(axis=1).map(lambda v:f'{v:.2%}' if pd.notna(v) else '—')
    view['Số tháng có nhóm'] = rates.notna().sum(axis=1)
    view['n/tháng TB'] = counts.mean(axis=1).round().astype('Int64')
    monthly_views[group] = view
    with pd.option_context('display.max_rows',None,'display.max_columns',None,'display.max_colwidth',None): display(view)
print(f'Ô: tỷ lệ (người mở / số khách). * Dưới {MIN_GROUP_N} khách; — không có nhóm. TB là trung bình đều các tháng có nhóm.')


## Đánh giá dự đoán riêng R, hai cách F và M_n_products
Mỗi mức được chấm bằng tỷ lệ acquisition lịch sử, chỉ dùng kết quả đến tháng chốt. Không dùng tỷ lệ của tháng đang test để xếp hạng. Mức chưa có lịch sử dùng baseline lịch sử. Cùng tập khách và cùng tie-break ngẫu nhiên tái lập như các cách M. AP/AUC/Precision/Recall/Lift càng cao càng tốt; Brier càng thấp càng tốt.


In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

# Tỷ lệ lịch sử của từng mức là điểm dự đoán; không áp đặt đơn điệu.
UNIVARIATE_METHODS={
    'R_group':'R', 'F_group':'F_event_months',
    'F_events_group':'F_events', 'M_group':'M_n_products'}
METRIC_COLS=['ROC_AUC','Average_Precision','Precision10','Recall10','Lift10','Brier']

def prediction_metrics(y,score):
    y=np.asarray(y,dtype=int); score=np.asarray(score,dtype=float)
    assert len(y)>0 and np.isfinite(score).all()
    tie=np.random.default_rng(42).random(len(y))
    k=max(1,int(len(y)*.10));top=np.lexsort((tie,-score))[:k]
    base=y.mean()
    return {'ROC_AUC':roc_auc_score(y,score) if len(np.unique(y))==2 else np.nan,
        'Average_Precision':average_precision_score(y,score) if y.any() else np.nan,
        'Precision10':y[top].mean(),'Recall10':y[top].sum()/y.sum() if y.any() else np.nan,
        'Lift10':y[top].mean()/base if base>0 else np.nan,
        'Brier':np.mean((score-y)**2)}

# Chỉ giữ thống kê gộp, không lưu toàn bộ panel lịch sử.
history_counts=[];history_totals=[]
for source in sorted(int(m) for m in months):
    if source+1 not in set(months) or source>=max(eval_cutoffs) or source<=min(months):continue
    month=pd.Timestamp(year=(source-1)//12,month=(source-1)%12+1,day=1).strftime('%Y-%m')
    if month in eval_months:
        hp=response_panel.loc[response_panel.eval_month.eq(month)].copy()
    else:
        hp,_=make_response_panel(df,events,PRODUCT_COLS,source)
        hp=add_groups(hp)
        hp['F_events_group']=hp.F_events.astype(int).astype(str)
    history_totals.append({'source':source,'result':source+1,'n':len(hp),'acquired':int(hp.target_next_month.sum())})
    for group,method in UNIVARIATE_METHODS.items():
        tmp=hp.groupby(group,observed=True).target_next_month.agg(n='size',acquired='sum').reset_index()
        tmp['level']=tmp[group].astype(str);tmp['method']=method;tmp['result']=source+1
        history_counts.append(tmp[['method','level','n','acquired','result']])
history_counts=pd.concat(history_counts,ignore_index=True)
history_totals=pd.DataFrame(history_totals)
univariate_rows=[];univariate_weights=[]
for cutoff in eval_cutoffs:
    month=pd.Timestamp(year=(cutoff-1)//12,month=(cutoff-1)%12+1,day=1).strftime('%Y-%m')
    p=response_panel.loc[response_panel.eval_month.eq(month)]
    total=history_totals.loc[history_totals.result.le(cutoff)]
    assert not total.empty and total.n.sum()>0
    historical_base=total.acquired.sum()/total.n.sum()
    meta={'eval_month':month,'result_month':(pd.Period(month,freq='M')+1).strftime('%Y-%m'),
        'n':len(p),'baseline':p.target_next_month.mean()}
    for group,method in UNIVARIATE_METHODS.items():
        h=history_counts.loc[history_counts.result.le(cutoff)&history_counts.method.eq(method)].groupby('level')[['n','acquired']].sum()
        h['rate']=h.acquired/h.n
        mapped=p[group].astype(str).map(h.rate)
        score=mapped.fillna(historical_base).to_numpy()
        univariate_rows.append(dict(meta,method=method,n_unseen=int(mapped.isna().sum()),
            **prediction_metrics(p.target_next_month,score)))
        weights=h.reset_index();weights['method']=method;weights['eval_month']=month;univariate_weights.append(weights)
    univariate_rows.append(dict(meta,method='Tỷ lệ chung lịch sử',n_unseen=0,
        **prediction_metrics(p.target_next_month,np.full(len(p),historical_base))))
univariate_metrics=pd.DataFrame(univariate_rows)
univariate_weight_table=pd.concat(univariate_weights,ignore_index=True)
assert univariate_metrics.eval_month.nunique()==N_EVAL_MONTHS
print('So sánh đơn biến: TB metric qua 5 tháng, mỗi tháng trọng số bằng nhau.')
display(univariate_metrics.groupby('method')[METRIC_COLS].mean().round(4))
print('Metrics từng tháng; mức chưa có lịch sử dùng tỷ lệ chung lịch sử:')
for metric in METRIC_COLS:
    view=univariate_metrics.pivot(index='method',columns='eval_month',values=metric).reindex(columns=eval_months)
    view.columns=[f'{m} → {(pd.Period(m,freq="M")+1).strftime("%Y-%m")}' for m in view.columns]
    view['TB']=view.mean(axis=1)
    print(metric);display(view.round(4))
display(univariate_metrics[['eval_month','method','n','n_unseen']])


## M có trọng số — chỉ phân tích riêng M
Hệ số sản phẩm được tính từ lịch sử có kết quả đến tháng chốt. Điểm là tổng hệ số các sản phẩm đang sở hữu ×100; không phải xác suất. Các khoảng điểm có độ rộng chính xác 5: (0,5], (5,10], …; điểm 0 riêng. M_n_products phía trên vẫn là số lượng sản phẩm, mỗi mức 1 sản phẩm.


In [ ]:
MIN_M_CUSTOMERS = 1000
MIN_M_OPENERS = 20
# Tính số khách sở hữu và số người mở thêm cho mọi cặp tháng.
# Chỉ dùng khách đủ 24 cờ sản phẩm ở cả hai tháng liên tiếp.
def build_weight_history(snapshots, product_cols):
    available_months = sorted(snapshots.month_idx.unique())
    month_set = set(available_months)
    rows = []

    for cutoff in available_months:
        if cutoff + 1 not in month_set:
            continue

        current = (
            snapshots.loc[
                snapshots.month_idx.eq(cutoff),
                ['customer_id'] + product_cols
            ]
            .set_index('customer_id')
        )
        following = (
            snapshots.loc[
                snapshots.month_idx.eq(cutoff + 1),
                ['customer_id'] + product_cols
            ]
            .set_index('customer_id')
        )

        valid_current = current.index[
            current.notna().all(axis=1)
        ]
        valid_following = following.index[
            following.notna().all(axis=1)
        ]
        ids = valid_current.intersection(valid_following)

        if len(ids) == 0:
            continue

        before = current.loc[ids, product_cols].to_numpy()
        after = following.loc[ids, product_cols].to_numpy()

        owns = before == 1
        # Một khách mở thêm nhiều sản phẩm vẫn tính một người.
        opened_any = ((before == 0) & (after == 1)).any(axis=1)

        n_owned = owns.sum(axis=0)
        n_openers = (owns & opened_any[:, None]).sum(axis=0)

        for j, product in enumerate(product_cols):
            rows.append({
                'source_cutoff': int(cutoff),
                'result_month_idx': int(cutoff + 1),
                'product': product,
                'n_owned': int(n_owned[j]),
                'acquired': int(n_openers[j]),
            })

    return pd.DataFrame(rows)


weight_history_all = build_weight_history(df, PRODUCT_COLS)


# Giữ tương thích với các dòng gọi hàm trong mục 8.2:
# get_product_weights(prior) và get_product_weights(latest_month)
def get_product_weights(source_month):
    period = pd.Period(source_month, freq='M')
    source_cutoff = period.year * 12 + period.month

    # Ví dụ source_month = 2016-03:
    # dùng mọi cặp từ đầu dữ liệu đến 03 → 04.
    history = weight_history_all.loc[
        weight_history_all.result_month_idx.le(source_cutoff + 1)
    ]

    if history.empty:
        return None

    weights = (
        history.groupby('product')[['n_owned', 'acquired']]
        .sum()
        .reindex(PRODUCT_COLS, fill_value=0)
    )

    # Tỷ lệ gộp: tổng người mở / tổng lượt khách sở hữu.
    # Không lấy trung bình tỷ lệ của các tháng.
    weights['rate'] = (
        weights.acquired
        / weights.n_owned.replace(0, np.nan)
    )

    # Chưa từng quan sát người sở hữu → chưa ước lượng được;
    # tạm đóng góp 0 điểm và đánh dấu rõ.
    weights['no_history'] = weights.n_owned.eq(0)
    weights['rate'] = weights.rate.fillna(0)

    # Chỉ đánh dấu để đọc biểu đồ, không thay đổi hệ số.
    weights['small_group'] = weights.n_owned.lt(MIN_M_CUSTOMERS)
    weights['few_openers'] = weights.acquired.lt(MIN_M_OPENERS)
    weights['use_default'] = False
    weights['rate_raw'] = weights['rate']

    return weights

In [ ]:
SCORE_STEP = 5
weighted_parts, weight_history = [], []
for month,p in response_panel.groupby('eval_month',sort=True):
    prior = (pd.Period(month,freq='M')-1).strftime('%Y-%m')
    weights = get_product_weights(prior)
    assert weights is not None, f'Thiếu lịch sử cho {month}'
    weights = weights.reindex(PRODUCT_COLS)
    period = pd.Period(month,freq='M'); cutoff = period.year*12+period.month
    current = df.loc[df.month_idx.eq(cutoff)].set_index('customer_id').loc[p.customer_id,PRODUCT_COLS]
    p=p.copy(); p['M_weighted'] = current.to_numpy(dtype=float) @ weights.rate.to_numpy(dtype=float)
    p['M_points']=(p.M_weighted*100).round(10)
    weighted_parts.append(p)
    history=weights.reset_index(); history['score_cutoff']=month; weight_history.append(history)
weighted_panel=pd.concat(weighted_parts,ignore_index=True)
score=weighted_panel.M_points
assert np.isfinite(score).all() and score.ge(0).all()
upper=max(SCORE_STEP,int(np.ceil(score.max()/SCORE_STEP))*SCORE_STEP)
weighted_edges=np.arange(0,upper+SCORE_STEP,SCORE_STEP)
assert np.all(np.diff(weighted_edges)==5)
weighted_labels=[f'({a:g}, {b:g}]' for a,b in zip(weighted_edges[:-1],weighted_edges[1:])]
WEIGHTED_SCORE_LABELS=['0']+weighted_labels
weighted_panel['M_weighted_group']=(pd.cut(score,weighted_edges,labels=weighted_labels,right=True)
    .cat.add_categories(['0']).mask(score.eq(0),'0').cat.reorder_categories(WEIGHTED_SCORE_LABELS,ordered=True))
assert weighted_panel.M_weighted_group.notna().all()
weighted_response=rate_table(weighted_panel,'M_weighted_group')
rates=weighted_response.pivot(index='M_weighted_group',columns='eval_month',values='rate').reindex(index=WEIGHTED_SCORE_LABELS,columns=eval_months)
counts=weighted_response.pivot(index='M_weighted_group',columns='eval_month',values='n').reindex(index=WEIGHTED_SCORE_LABELS,columns=eval_months)
opened=weighted_response.pivot(index='M_weighted_group',columns='eval_month',values='acquired').reindex(index=WEIGHTED_SCORE_LABELS,columns=eval_months)
weighted_view=pd.DataFrame(index=rates.index)
for m in eval_months:
    label=f'{m} → {(pd.Period(m,freq="M")+1).strftime("%Y-%m")}'
    weighted_view[label]=[f'{r:.2%}'+(' *' if n<MIN_GROUP_N else '') if pd.notna(r) else '—' for r,n in zip(rates[m],counts[m])]
weighted_view['TB tỷ lệ']=rates.mean(axis=1).map(lambda v:f'{v:.2%}' if pd.notna(v) else '—')
weighted_view['Số tháng có nhóm']=rates.notna().sum(axis=1)
weighted_view['n/tháng TB']=counts.mean(axis=1)
# Chỉ sắp bảng theo số khách; biểu đồ vẫn theo khoảng điểm.
weighted_view=weighted_view.loc[rates.notna().any(axis=1)].copy()
weighted_view=weighted_view.sort_values('n/tháng TB',ascending=False,kind='stable')
weighted_view['n/tháng TB']=weighted_view['n/tháng TB'].round().astype('Int64')
print('* Dưới 100 khách. Số khách và người mở từng tháng lưu trong CSV chi tiết.')
with pd.option_context('display.max_rows',None,'display.max_columns',None): display(weighted_view)
plot=rates.mean(axis=1).dropna()
fig,ax=plt.subplots(figsize=(12,4.8)); x=np.arange(len(plot))
bars=ax.bar(x,plot.to_numpy()*100,color=CHART_BLUE,width=.7)
ax.bar_label(bars,labels=[f'{r:.1%}' if r>0 else '' for r in plot],padding=3,fontsize=8,rotation=90)
ax.axhline(baseline_mean*100,color=CHART_BASELINE,linestyle='--',label=f'Tỷ lệ chung TB: {baseline_mean:.2%}')
labels=[str(v).replace('(','').replace(']','').replace(', ','–') for v in plot.index]
ax.set_xticks(x); ax.set_xticklabels(labels,rotation=45,ha='right',fontsize=9)
ax.set_ylim(0,max(float(plot.max()),float(baseline_mean))*100*1.3 if plot.max()>0 else 1)
ax.set_ylabel('Tỷ lệ acquisition trung bình (%)'); ax.set_xlabel('Điểm M có trọng số — khoảng cách 5 điểm')
ax.set_title(f'M có trọng số | trung bình {len(eval_months)} lần đánh giá')
ax.legend(frameon=False); ax.grid(False,axis='x'); plt.tight_layout(); plt.show()


## Các cách chấm M bổ sung — không kết hợp R hoặc F
- **M theo sản phẩm đang sở hữu:** phần trên cộng tỷ lệ lịch sử của khách có từng sản phẩm, có thể đang có nhiều sản phẩm.
- **M riêng lẻ:** hệ số của sản phẩm A lấy từ khách chỉ sở hữu đúng A; điểm khách là tổng các hệ số. Bản cộng thô là điểm, không phải xác suất.
- **M theo danh mục:** tỷ lệ lịch sử của đúng tổ hợp sản phẩm. Bảng bucket chỉ dùng danh mục có ít nhất MIN_HISTORY lượt lịch sử; báo cáo độ phủ.
- **M riêng lẻ hiệu chỉnh:** isotonic chuyển điểm cộng thành tỷ lệ ước lượng.
- **M kết hợp:** ưu tiên danh mục đủ mẫu hoặc trộn tỷ lệ danh mục với tỷ lệ riêng lẻ hiệu chỉnh.

Mọi hệ số và isotonic tại tháng t chỉ dùng cặp có tháng kết quả ≤ t. Chạy lại cho cả 5 tháng. Các bảng điểm dùng khoảng cách 5 điểm, 0 riêng; biểu đồ giữ thứ tự điểm, bảng sắp theo số khách. Đây là các cách chấm M, không phải RF/RFM. Isotonic fit cùng lịch sử, chưa có calibration bằng dự đoán ngoài mẫu; các ngưỡng chưa được tuning trên tập độc lập.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score

MIN_HISTORY = 1000
K_SHRINK = 500

BUCKET_STEPS = {
    'M riêng lẻ cộng thô': 5,
    'M riêng lẻ hiệu chỉnh': 2,
    'M theo danh mục': 5,
    'M kết hợp ngưỡng cứng': 5,
    'M kết hợp làm mượt': 5
}

METRIC_COLS = [
    'ROC_AUC',
    'Average_Precision',
    'Precision10',
    'Recall10',
    'Lift10',
    'Brier'
]

P = len(PRODUCT_COLS)
assert P <= 63

SHIFTS = np.arange(P, dtype=np.uint64)
BITS = np.uint64(1) << SHIFTS


def month_label(idx):
    year, month_zero = divmod(int(idx) - 1, 12)
    return f'{year}-{month_zero + 1:02d}'


def pair_arrays(source):
    """Khách đủ cờ sản phẩm tại tháng source và source+1."""
    cols = ['customer_id'] + PRODUCT_COLS

    before = (
        df.loc[df.month_idx.eq(source), cols]
        .set_index('customer_id')
    )
    after = (
        df.loc[df.month_idx.eq(source + 1), cols]
        .set_index('customer_id')
    )

    ids = before.index[
        before.notna().all(axis=1)
    ].intersection(
        after.index[after.notna().all(axis=1)]
    )

    x = before.loc[ids, PRODUCT_COLS].to_numpy()
    z = after.loc[ids, PRODUCT_COLS].to_numpy()

    owns = x == 1
    opened = ((x == 0) & (z == 1)).any(axis=1)

    return ids, owns, opened


def to_code(owns):
    return owns.astype(np.uint64) @ BITS


def code_name(code):
    names = [
        product.removeprefix('prod_')
        for j, product in enumerate(PRODUCT_COLS)
        if (int(code) >> j) & 1
    ]
    return ' + '.join(names) or 'Không có sản phẩm'


# ----------------------------------------------------------
# 1. Thống kê lịch sử theo danh mục.
# ----------------------------------------------------------
months_all = sorted(df.month_idx.unique())
month_set = set(months_all)
count_parts = {}

for source in months_all:
    if (
        source + 1 not in month_set
        or source + 1 > max(eval_cutoffs)
    ):
        continue

    _, owns_h, opened_h = pair_arrays(source)

    count_parts[int(source)] = (
        pd.DataFrame({
            'code': to_code(owns_h),
            'opened': opened_h.astype('int8')
        })
        .groupby('code')['opened']
        .agg(n='size', k='sum')
    )


# ----------------------------------------------------------
# 2. Chấm điểm từng tháng, không học từ kết quả tháng test.
# ----------------------------------------------------------
m_score_parts = []
m_metric_rows = []
m_coverage_rows = []
m_single_weight_parts = []

for cutoff in eval_cutoffs:
    historical_parts = [
        table
        for source, table in count_parts.items()
        if source + 1 <= cutoff
    ]
    assert historical_parts, f'Thiếu lịch sử tại {cutoff}'

    h = (
        pd.concat(historical_parts)
        .groupby(level=0)[['n', 'k']]
        .sum()
    )
    assert h['n'].sum() > 0

    h['rate'] = h['k'] / h['n']
    historical_base = h['k'].sum() / h['n'].sum()

    # Hệ số từ khách chỉ sở hữu đúng một sản phẩm.
    single = h.reindex(BITS)
    raw_weights = single['rate'].to_numpy()

    weights = np.where(
        np.isnan(raw_weights),
        historical_base,
        raw_weights
    )

    weight_table = pd.DataFrame({
        'product': PRODUCT_COLS,
        'history_n': single['n'].fillna(0).to_numpy(),
        'history_opened': single['k'].fillna(0).to_numpy(),
        'points': weights * 100,
        'uses_baseline': np.isnan(raw_weights),
        'eval_month': month_label(cutoff)
    })
    m_single_weight_parts.append(weight_table)

    # Hiệu chỉnh điểm cộng bằng dữ liệu lịch sử.
    historical_codes = h.index.to_numpy(dtype=np.uint64)
    historical_owns = (
        (historical_codes[:, None] >> SHIFTS) & np.uint64(1)
    ).astype(float)

    historical_additive = historical_owns @ weights

    calibrator = IsotonicRegression(
        y_min=0,
        y_max=1,
        out_of_bounds='clip'
    )
    calibrator.fit(
        historical_additive,
        h['rate'],
        sample_weight=h['n']
    )

    # Dữ liệu đánh giá: tháng chốt → tháng sau.
    ids, owns, y = pair_arrays(cutoff)
    assert len(y) > 0

    current_codes = pd.Series(to_code(owns))
    group_n = current_codes.map(h['n']).fillna(0).to_numpy()
    group_rate = current_codes.map(h['rate']).to_numpy()

    group_default = np.where(
        np.isnan(group_rate),
        historical_base,
        group_rate
    )
    group_zero = np.nan_to_num(group_rate)

    additive = owns.astype(float) @ weights
    calibrated = calibrator.predict(additive)

    missing_single = (
        owns & np.isnan(raw_weights)[None, :]
    ).any(axis=1)

    methods = {
        'M riêng lẻ cộng thô': (
            additive, False
        ),
        'M riêng lẻ hiệu chỉnh': (
            calibrated, True
        ),
        'M theo danh mục': (
            group_default, True
        ),
        'M kết hợp ngưỡng cứng': (
            np.where(
                group_n >= MIN_HISTORY,
                group_zero,
                calibrated
            ),
            True
        ),
        'M kết hợp làm mượt': (
            (
                group_n * group_zero
                + K_SHRINK * calibrated
            ) / (group_n + K_SHRINK),
            True
        )
    }

    tie = np.random.default_rng(42).random(len(y))
    top_k = max(1, int(len(y) * 0.10))
    baseline = y.mean()

    for method, (scores, is_probability) in methods.items():
        scores = np.asarray(scores, dtype=float)
        assert np.isfinite(scores).all()

        # Danh mục ít lịch sử không đưa vào bảng bucket.
        # Metrics vẫn tính trên toàn bộ khách.
        display_score = scores.copy()

        if method == 'M theo danh mục':
            display_score[group_n < MIN_HISTORY] = np.nan

        m_score_parts.append(
            pd.DataFrame({
                'customer_id': ids.to_numpy(),
                'eval_month': month_label(cutoff),
                'method': method,
                'M_points': np.round(display_score * 100, 10),
                'target_next_month': y.astype('int8')
            })
        )

        top = np.lexsort((tie, -scores))[:top_k]

        m_metric_rows.append({
            'eval_month': month_label(cutoff),
            'result_month': month_label(cutoff + 1),
            'method': method,
            'n': len(y),
            'baseline': baseline,
            'ROC_AUC': (
                roc_auc_score(y, scores)
                if len(np.unique(y)) == 2 else np.nan
            ),
            'Average_Precision': (
                average_precision_score(y, scores)
                if y.any() else np.nan
            ),
            'Precision10': y[top].mean(),
            'Recall10': (
                y[top].sum() / y.sum()
                if y.any() else np.nan
            ),
            'Lift10': (
                y[top].mean() / baseline
                if baseline > 0 else np.nan
            ),
            'Brier': (
                np.mean((scores - y) ** 2)
                if is_probability else np.nan
            )
        })

        m_coverage_rows.append({
            'eval_month': month_label(cutoff),
            'method': method,
            'n_total': len(y),
            'n_in_bucket': int(np.isfinite(display_score).sum()),
            'n_excluded': int(np.isnan(display_score).sum()),
            'n_missing_single_weight': int(missing_single.sum())
        })

    if cutoff == max(eval_cutoffs):
        m_top_portfolios = (
            pd.DataFrame({
                'code': to_code(owns),
                'opened': y.astype('int8')
            })
            .groupby('code')['opened']
            .agg(n='size', acquired='sum')
            .sort_values('n', ascending=False)
            .head(10)
        )

        m_top_portfolios['history_n'] = (
            m_top_portfolios.index.map(h['n']).fillna(0)
        )
        m_top_portfolios['history_rate'] = (
            m_top_portfolios.index.map(h['rate'])
        )
        m_top_portfolios['actual_rate'] = (
            m_top_portfolios['acquired']
            / m_top_portfolios['n']
        )
        m_top_portfolios.index = [
            code_name(code) for code in m_top_portfolios.index
        ]


m_scores = pd.concat(m_score_parts, ignore_index=True)
m_single_weights = pd.concat(
    m_single_weight_parts, ignore_index=True
)
m_metrics = pd.DataFrame(m_metric_rows)
m_coverage = pd.DataFrame(m_coverage_rows)

assert m_metrics['eval_month'].nunique() == N_EVAL_MONTHS


# ----------------------------------------------------------
# 3. Độ phủ và danh mục đông khách.
# ----------------------------------------------------------
print('Độ phủ trung bình qua các tháng:')

coverage_summary = (
    m_coverage.groupby('method')[
        [
            'n_total',
            'n_in_bucket',
            'n_excluded',
            'n_missing_single_weight'
        ]
    ]
    .mean()
    .round()
    .astype('Int64')
)

coverage_summary['Độ phủ TB'] = (
    m_coverage.assign(
        coverage=m_coverage['n_in_bucket'] / m_coverage['n_total']
    )
    .groupby('method')['coverage']
    .mean()
    .map(lambda value: f'{value:.2%}')
)
display(coverage_summary)

print('10 danh mục đông khách nhất ở tháng chốt cuối:')

v = m_top_portfolios.copy()

for col in ['history_rate', 'actual_rate']:
    v[col] = v[col].map(
        lambda value: f'{value:.2%}' if pd.notna(value) else '—'
    )

display(v)


# ----------------------------------------------------------
# 4. Bucket riêng cho từng phương pháp.
#    Riêng lẻ hiệu chỉnh: 2 điểm.
#    Các phương pháp còn lại: 5 điểm.
#    Cùng phương pháp dùng cùng biên qua mọi tháng.
# ----------------------------------------------------------
METHOD_LEVELS = {}
METHOD_EDGES = {}
bucket_parts = []

for method in m_metrics['method'].unique():
    part = m_scores.loc[
        m_scores['method'].eq(method)
    ].copy()

    points = part['M_points']
    valid_points = points.dropna()
    step = BUCKET_STEPS[method]

    assert np.isfinite(valid_points).all()
    assert valid_points.ge(0).all()

    max_points = (
        float(valid_points.max())
        if not valid_points.empty else 0
    )
    upper = max(
        step,
        int(np.ceil(max_points / step)) * step
    )

    edges = np.arange(0, upper + step, step)
    labels = [
        f'({left:g}, {right:g}]'
        for left, right in zip(edges[:-1], edges[1:])
    ]
    levels = ['0'] + labels

    METHOD_EDGES[method] = edges
    METHOD_LEVELS[method] = levels

    buckets = (
        pd.cut(
            points,
            bins=edges,
            labels=labels,
            right=True
        )
        .cat.add_categories(['0'])
        .mask(points.eq(0), '0')
        .cat.reorder_categories(levels, ordered=True)
    )

    assert buckets.loc[points.notna()].notna().all()

    # Các phương pháp có bộ nhãn khác nhau.
    part['M_bucket'] = buckets.astype(object)
    bucket_parts.append(part)

m_scores = pd.concat(bucket_parts, ignore_index=True)


# ----------------------------------------------------------
# 5. Tỷ lệ acquisition theo bucket và tháng.
# ----------------------------------------------------------
m_method_response = (
    m_scores
    .groupby(
        ['method', 'eval_month', 'M_bucket'],
        observed=True
    )
    .agg(
        n=('target_next_month', 'size'),
        acquired=('target_next_month', 'sum')
    )
    .reset_index()
)

m_method_response['rate'] = (
    m_method_response['acquired'] / m_method_response['n']
)

m_method_views = {}

for method in m_metrics['method'].unique():
    levels = METHOD_LEVELS[method]
    step = BUCKET_STEPS[method]

    table = m_method_response.loc[
        m_method_response['method'].eq(method)
    ]

    rates = (
        table.pivot(
            index='M_bucket',
            columns='eval_month',
            values='rate'
        )
        .reindex(index=levels, columns=eval_months)
    )
    counts = (
        table.pivot(
            index='M_bucket',
            columns='eval_month',
            values='n'
        )
        .reindex(index=levels, columns=eval_months)
    )

    view = pd.DataFrame(index=rates.index)

    for month in eval_months:
        result_month = (
            pd.Period(month, freq='M') + 1
        ).strftime('%Y-%m')

        view[f'{month} → {result_month}'] = [
            (
                f'{rate:.2%}'
                + (' *' if count < MIN_GROUP_N else '')
            )
            if pd.notna(rate) else '—'
            for rate, count in zip(rates[month], counts[month])
        ]

    view['TB tỷ lệ'] = rates.mean(axis=1).map(
        lambda value: f'{value:.2%}' if pd.notna(value) else '—'
    )
    view['Số tháng có nhóm'] = rates.notna().sum(axis=1)
    view['n/tháng TB'] = (
        counts.mean(axis=1).round().astype('Int64')
    )

    # Ẩn hàng không có mẫu trong tất cả các tháng.
    view = view.loc[rates.notna().any(axis=1)].copy()

    # Chỉ bảng sắp theo số khách giảm dần.
    view = view.sort_values(
        'n/tháng TB',
        ascending=False,
        kind='stable'
    )
    m_method_views[method] = view

    print(f'\n{method} — bucket {step} điểm')

    with pd.option_context(
        'display.max_rows', None,
        'display.max_columns', None
    ):
        display(view)

    # Biểu đồ giữ thứ tự khoảng điểm tăng dần.
    plot = rates.mean(axis=1).dropna()

    if plot.empty:
        print('Không có bucket đủ dữ liệu để vẽ.')
        continue

    fig, ax = plt.subplots(figsize=(12, 4.8))
    x = np.arange(len(plot))

    bars = ax.bar(
        x,
        plot.to_numpy() * 100,
        color=CHART_BLUE,
        width=.65
    )

    ax.bar_label(
        bars,
        labels=[
            f'{rate:.1%}' if rate > 0 else ''
            for rate in plot
        ],
        padding=3,
        fontsize=8,
        rotation=90
    )

    labels = [
        str(label)
        .replace('(', '')
        .replace(']', '')
        .replace(', ', '–')
        for label in plot.index
    ]

    ax.set_xticks(x)
    ax.set_xticklabels(
        labels,
        rotation=45,
        ha='right',
        fontsize=9
    )

    ax.axhline(
        baseline_mean * 100,
        color=CHART_BASELINE,
        linestyle='--',
        label=f'Tỷ lệ chung TB: {baseline_mean:.2%}'
    )

    ymax = max(
        float(plot.max()) * 100,
        float(baseline_mean) * 100,
        1
    )
    ax.set_ylim(0, ymax * 1.3)

    ax.set_xlabel(f'Khoảng điểm M — mỗi khoảng {step} điểm')
    ax.set_ylabel('Tỷ lệ acquisition trung bình (%)')
    ax.set_title(
        f'{method} | trung bình {N_EVAL_MONTHS} lần đánh giá'
    )
    ax.legend(frameon=False)
    ax.grid(False, axis='x')

    plt.tight_layout()
    plt.show()

print(
    f'* Dưới {MIN_GROUP_N} khách; — không có mẫu. '
    'TB là trung bình đều tỷ lệ các tháng có nhóm.'
)


# ----------------------------------------------------------
# 6. Hệ số sản phẩm riêng lẻ.
# ----------------------------------------------------------
w = (
    m_single_weights.loc[
        m_single_weights['eval_month'].eq(latest_month)
    ]
    .sort_values('product')
)

fig, ax = plt.subplots(figsize=(11, 9))

colors = np.where(
    w['history_n'].lt(MIN_HISTORY) | w['uses_baseline'],
    '#D88935',
    CHART_BLUE
)

bars = ax.barh(
    w['product'].str.replace('prod_', '', regex=False),
    w['points'],
    color=colors
)

ax.bar_label(
    bars,
    labels=[f'{point:.2f}' for point in w['points']],
    fontsize=8,
    padding=3
)

ax.set_xlim(0, max(float(w['points'].max()) * 1.2, 1))
ax.set_title(
    'M riêng lẻ: hệ số từ khách chỉ có đúng một sản phẩm'
)
ax.set_xlabel(
    'Điểm sản phẩm (%); cam: ít mẫu hoặc dùng tỷ lệ chung'
)

plt.tight_layout()
plt.show()


# ----------------------------------------------------------
# 7. Metrics từ điểm gốc, không phụ thuộc cách chia bucket.
# ----------------------------------------------------------
print('AP và Lift top 10% theo từng tháng:')

for metric in ['Average_Precision', 'Lift10']:
    monthly_metric = (
        m_metrics.pivot(
            index='method',
            columns='eval_month',
            values=metric
        )
        .reindex(columns=eval_months)
    )

    monthly_metric.columns = [
        f'{month} → '
        f'{(pd.Period(month, freq="M") + 1).strftime("%Y-%m")}'
        for month in monthly_metric.columns
    ]

    print(metric)
    display(monthly_metric.round(4))

print('Trung bình metrics, mỗi tháng trọng số bằng nhau:')

display(
    m_metrics.groupby('method')[METRIC_COLS]
    .mean()
    .round(4)
)

## 9. Lưu output
Giữ hai output cũ; bổ sung bảng audit và tỷ lệ acquisition để báo cáo. Không lưu bảng panel lớn mặc định.

In [ ]:
assert events.customer_id.isin(window.customer_id).all()
assert events['product'].isin(PRODUCT_COLS).all() and events.record_date.le(LAST_MONTH).all()

os.makedirs(WORK_DIR, exist_ok=True)
events.drop(columns=['month_idx', 'row']).to_parquet(os.path.join(WORK_DIR, 'acquisition_events.parquet'), index=False)
rfm.drop(columns=['tenure_bucket'], errors='ignore').to_parquet(os.path.join(WORK_DIR, 'rfm_proxy_raw.parquet'), index=False)
print(f'Đã lưu: {len(events):,} event | {len(rfm):,} RFM '
      f'(as_of={AS_OF.date()}, giữ NaN + cờ) → {WORK_DIR}')

In [ ]:
os.makedirs(WORK_DIR,exist_ok=True)
pd.DataFrame(audits).to_csv(os.path.join(WORK_DIR,'response_audit_5_months.csv'),index=False)
for group,table in response_tables.items():
    table.to_csv(os.path.join(WORK_DIR,f'acquisition_rate_{group}.csv'),index=False)
    monthly_views[group].to_csv(os.path.join(WORK_DIR,f'monthly_comparison_{group}.csv'))
    mean_tables[group].to_csv(os.path.join(WORK_DIR,f'mean_acquisition_{group}.csv'))
weighted_response.to_csv(os.path.join(WORK_DIR,'acquisition_rate_weighted_M.csv'),index=False)
weighted_view.to_csv(os.path.join(WORK_DIR,'weighted_M_monthly_comparison.csv'))
pd.concat(weight_history,ignore_index=True).to_csv(os.path.join(WORK_DIR,'weighted_M_coefficients.csv'),index=False)
with open(os.path.join(WORK_DIR,'evaluation_config.json'),'w') as f:
    import json
    json.dump({'N_EVAL_MONTHS':N_EVAL_MONTHS,'eval_months':eval_months,'SCORE_STEP':SCORE_STEP,'weighted_edges':weighted_edges.tolist(),'MIN_GROUP_N':MIN_GROUP_N},f,indent=2)
print('Đã lưu bảng tỷ lệ từng tháng, mẫu số và TB cho 5 lần đánh giá.')


In [ ]:
m_metrics.to_csv(os.path.join(WORK_DIR,'M_methods_metrics_5_months.csv'),index=False)
m_coverage.to_csv(os.path.join(WORK_DIR,'M_methods_coverage.csv'),index=False)
m_method_response.to_csv(os.path.join(WORK_DIR,'M_methods_acquisition_by_month.csv'),index=False)
m_single_weights.to_csv(os.path.join(WORK_DIR,'M_single_product_weights.csv'),index=False)
m_top_portfolios.to_csv(os.path.join(WORK_DIR,'M_top_portfolios_latest.csv'))
for i,(method,view) in enumerate(m_method_views.items(),1):
    view.to_csv(os.path.join(WORK_DIR,f'M_method_{i}_monthly_comparison.csv'))


In [ ]:
univariate_metrics.to_csv(os.path.join(WORK_DIR,'univariate_metrics_5_months.csv'),index=False)
univariate_weight_table.to_csv(os.path.join(WORK_DIR,'univariate_historical_weights.csv'),index=False)
all_method_metrics=pd.concat([univariate_metrics,m_metrics],ignore_index=True)
all_method_metrics.to_csv(os.path.join(WORK_DIR,'all_methods_metrics_5_months.csv'),index=False)
print('So sánh chung: các biến riêng và các cách chấm M trên cùng 5 tháng.')
display(all_method_metrics.groupby('method')[METRIC_COLS].mean().round(4))


## Giới hạn
5 cặp tháng gần nhất của dữ liệu đến 2016-05 là 2015-12→2016-01, 2016-01→02, 2016-02→03, 2016-03→04 và 2016-04→05. Đây là dự đoán từng tháng, không phải target 5 tháng.

TB là trung bình đều tỷ lệ các tháng có nhóm; không có nhóm giữ thiếu. Cùng khách có thể xuất hiện nhiều tháng. Điều kiện đủ dữ liệu tháng sau xác định mẫu đánh giá; cần báo cáo khách bị loại. F phụ thuộc độ dài lịch sử quan sát. Không kết luận ổn định mọi năm chỉ từ 5 tháng. Chưa chạy toàn bộ dataset trong phiên sửa notebook này.


## Lưu tất cả biến thể R/F/M cho mọi khách ở mọi tháng
Mỗi tháng một file Parquet trong `rfm_all_months/`; tổng số dòng bằng tổng snapshot gốc. Có R, F/F_events, F_event_months, M_n_products, M_renta; điểm M theo sản phẩm sở hữu, riêng lẻ cộng thô, riêng lẻ hiệu chỉnh, theo danh mục, kết hợp cứng và làm mượt; hệ số lịch sử riêng R/F/M; bucket và cờ lịch sử.

Điểm tại tháng t chỉ dùng cặp có kết quả đến t. Tháng đầu chưa có lịch sử: điểm học từ target là NaN, không giả làm 0. Khách thiếu cờ sản phẩm vẫn được lưu, các điểm M không tính được giữ NaN. R thiếu nghĩa là chưa thấy event, không phải R=0. Bucket riêng lẻ hiệu chỉnh cách 2 điểm; các M khác cách 5 điểm. Hệ số sản phẩm, manifest và audit đi kèm. Không thêm target vào bảng feature; không lưu điểm mô tả dùng tương lai. Phần này chạy tất cả tháng, độc lập với 5 tháng đánh giá.


In [ ]:
from pathlib import Path
import json
from sklearn.isotonic import IsotonicRegression

# Từng partition chứa mọi khách có snapshot tại tháng đó, không chỉ khách eval.
ALL_MONTHS_DIR = Path(WORK_DIR) / 'rfm_all_months'
ALL_MONTHS_DIR.mkdir(parents=True, exist_ok=True)

def iter_all_month_features(snapshots, acquisition_events, product_cols):
    month_ids=sorted(int(m) for m in snapshots.month_idx.unique())
    p_count=len(product_cols)
    shifts=np.arange(p_count,dtype=np.uint64)
    bits=np.uint64(1)<<shifts
    assert p_count<=63
    history=pd.DataFrame(columns=['n','k'],dtype=float)
    owner_n=np.zeros(p_count,dtype=np.int64)
    owner_k=np.zeros(p_count,dtype=np.int64)
    variable_hist={name:pd.DataFrame(columns=['n','k'],dtype=float)
        for name in ['R','F_event_months','F_events','M_n_products']}
    previous=None; previous_idx=None
    for cutoff in month_ids:
        current=snapshots.loc[snapshots.month_idx.eq(cutoff)].set_index('customer_id')
        features=build_rfm(snapshots,acquisition_events,product_cols,cutoff).set_index('customer_id')
        features=features.loc[current.index].copy()
        # Cập nhật lịch sử bằng cặp tháng trước → tháng hiện tại.
        # Kết quả tháng hiện tại được dùng chấm cho tháng kế tiếp.
        if previous is not None and previous_idx+1==cutoff:
            valid_before=previous.index[previous[product_cols].notna().all(axis=1)]
            valid_now=current.index[current[product_cols].notna().all(axis=1)]
            ids=valid_before.intersection(valid_now)
            x=previous.loc[ids,product_cols].to_numpy()
            z=current.loc[ids,product_cols].to_numpy()
            owns=x==1; y=((x==0)&(z==1)).any(axis=1)
            codes=owns.astype(np.uint64)@bits
            part=pd.DataFrame({'code':codes,'opened':y.astype('int8')}).groupby('code').opened.agg(n='size',k='sum')
            history=history.add(part,fill_value=0)
            owner_n+=owns.sum(axis=0)
            owner_k+=(owns & y[:,None]).sum(axis=0)
            for name in variable_hist:
                levels=previous_features.loc[ids,name].astype('Int64').astype('string').fillna('No event')
                t=pd.DataFrame({'level':levels.to_numpy(),'opened':y.astype('int8')}).groupby('level').opened.agg(n='size',k='sum')
                variable_hist[name]=variable_hist[name].add(t,fill_value=0)
        out=features.reset_index()
        out['month_idx']=cutoff
        out['products_complete']=current[product_cols].notna().all(axis=1).to_numpy()
        out['is_onboarding_batch']=out.customer_id.map(COHORT).fillna(False).astype(bool)
        out['event_history_group']=np.where(out.has_event,'Has event','No event')
        # Giữ cả các biến gốc và mức nhóm dùng trong phân tích.
        out=add_groups(out)
        out['F_events_group']=out.F_events.astype(int).astype(str)
        values=current[product_cols].to_numpy()
        owns=values==1
        codes=owns.astype(np.uint64)@bits
        complete=out.products_complete.to_numpy()
        out['portfolio_code']=pd.array(codes,dtype='UInt64')
        out.loc[~complete,'portfolio_code']=pd.NA
        score_columns=['M_weighted','M_points','M_single_raw','M_single_points',
            'M_single_calibrated','M_single_calibrated_points','M_group_rate','M_group_points',
            'M_hard_rate','M_hard_points','M_smooth_rate','M_smooth_points',
            'R_historical_rate','F_event_months_historical_rate','F_events_historical_rate','M_n_products_historical_rate']
        for name in score_columns:out[name]=np.nan
        out['M_group_history_n']=np.zeros(len(out),dtype=np.int64)
        out['M_has_group']=False
        out['M_single_uses_baseline']=False
        out['M_weighted_missing_owner_history']=False
        out['score_history_available']=bool(len(history) and history.n.sum()>0)
        out['history_customer_months']=int(history.n.sum()) if len(history) else 0
        out['historical_baseline']=np.nan
        coeff=pd.DataFrame({'product':product_cols,'month_idx':cutoff,
            'owner_history_n':owner_n.copy(),'owner_history_opened':owner_k.copy()})
        coeff['owner_weight']=np.divide(owner_k,owner_n,out=np.zeros(p_count,dtype=float),where=owner_n>0)
        coeff['single_history_n']=0
        coeff['single_history_opened']=0
        coeff['single_weight']=np.nan
        coeff['single_uses_baseline']=True
        coeff['score_history_available']=out.score_history_available.iloc[0]
        if out.score_history_available.iloc[0]:
            h=history.copy();h['rate']=h.k/h.n
            base=h.k.sum()/h.n.sum();out['historical_baseline']=base
            single=h.reindex(bits)
            raw=single.rate.to_numpy();w=np.where(np.isnan(raw),base,raw)
            coeff['single_history_n']=single.n.fillna(0).to_numpy()
            coeff['single_history_opened']=single.k.fillna(0).to_numpy()
            coeff['single_weight']=w
            coeff['single_uses_baseline']=np.isnan(raw)
            historical_codes=h.index.to_numpy(dtype=np.uint64)
            additive_h=((historical_codes[:,None]>>shifts)&np.uint64(1)).astype(float)@w
            calibrator=IsotonicRegression(y_min=0,y_max=1,out_of_bounds='clip')
            calibrator.fit(additive_h,h.rate,sample_weight=h.n)
            add=owns.astype(float)@w
            calibrated=calibrator.predict(add)
            code_series=pd.Series(codes)
            gn=code_series.map(h.n).fillna(0).to_numpy()
            gr=code_series.map(h.rate).to_numpy()
            gd=np.where(np.isnan(gr),base,gr)
            hard=np.where(gn>=MIN_HISTORY,np.nan_to_num(gr),calibrated)
            smooth=(gn*np.nan_to_num(gr)+K_SHRINK*calibrated)/(gn+K_SHRINK)
            owner_score=owns.astype(float)@coeff.owner_weight.to_numpy()
            computed={'M_weighted':owner_score,'M_points':owner_score*100,
                'M_single_raw':add,'M_single_points':add*100,
                'M_single_calibrated':calibrated,'M_single_calibrated_points':calibrated*100,
                'M_group_rate':gd,'M_group_points':gd*100,
                'M_hard_rate':hard,'M_hard_points':hard*100,
                'M_smooth_rate':smooth,'M_smooth_points':smooth*100}
            for name,score in computed.items():
                out[name]=np.where(complete,np.round(score,10),np.nan)
            out['M_group_history_n']=np.where(complete,gn,0).astype(np.int64)
            out['M_has_group']=complete & (gn>=MIN_HISTORY)
            out['M_single_uses_baseline']=complete & (owns & np.isnan(raw)[None,:]).any(axis=1)
            out['M_weighted_missing_owner_history']=complete & (owns & (owner_n==0)[None,:]).any(axis=1)
            for name,vh in variable_hist.items():
                rates=vh.k/vh.n
                levels=out[name].astype('Int64').astype('string').fillna('No event')
                out[name+'_historical_rate']=levels.map(rates).fillna(base).astype(float)
                if name=='M_n_products':out.loc[~complete,name+'_historical_rate']=np.nan
        # Bucket M: hiệu chỉnh 2 điểm; các cách khác 5 điểm.
        # Khoảng cố định theo giá trị, không phụ thuộc max của tháng hay tương lai.
        bucket_specs={'M_points':5,'M_single_points':5,'M_single_calibrated_points':2,
            'M_group_points':5,'M_hard_points':5,'M_smooth_points':5}
        for score_col,step in bucket_specs.items():
            score=out[score_col]
            labels=pd.Series(pd.NA,index=out.index,dtype='string')
            labels.loc[score.eq(0)]='0'
            pos=score.gt(0)
            upper=np.ceil(score.loc[pos]/step).astype(int)*step
            lower=upper-step
            labels.loc[pos]=['('+str(a)+', '+str(b)+']' for a,b in zip(lower,upper)]
            out[score_col+'_bucket']=labels
        # Bucket danh mục đủ mẫu: riêng với bucket scorer dùng tất cả danh mục.
        out['M_group_points_bucket_supported']=out['M_group_points_bucket'].where(out.M_has_group)
        yield cutoff,out,coeff
        previous=current
        previous_features=features
        previous_idx=cutoff

monthly_export_audit=[]; coefficient_parts=[]
for cutoff,monthly,coefficients in iter_all_month_features(df,events,PRODUCT_COLS):
    month=month_label(cutoff)
    destination=ALL_MONTHS_DIR / f'rfm_{month}.parquet'
    assert not monthly.customer_id.duplicated().any()
    monthly.to_parquet(destination,index=False,compression='snappy')
    coefficient_parts.append(coefficients)
    monthly_export_audit.append({'month_idx':cutoff,'month':month,'n_customers':len(monthly),
        'n_has_event':int(monthly.has_event.sum()),'n_complete_products':int(monthly.products_complete.sum()),
        'n_with_M_score':int(monthly.M_points.notna().sum()),'file':destination.name})
    print(f'{month}: {len(monthly):,} khách → {destination.name}')
monthly_export_audit=pd.DataFrame(monthly_export_audit)
assert monthly_export_audit.n_customers.sum()==len(df)
assert len(monthly_export_audit)==df.month_idx.nunique()
monthly_export_audit.to_csv(ALL_MONTHS_DIR/'export_audit.csv',index=False)
pd.concat(coefficient_parts,ignore_index=True).to_parquet(ALL_MONTHS_DIR/'score_coefficients.parquet',index=False)
with open(ALL_MONTHS_DIR/'manifest.json','w',encoding='utf-8') as f:
    json.dump({'feature_files':monthly_export_audit.file.tolist(),'n_rows':int(len(df)),
        'history_rule':'result_month <= feature_month; no future descriptive scores',
        'MIN_HISTORY':MIN_HISTORY,'K_SHRINK':K_SHRINK,
        'bucket_steps':{'M_single_calibrated_points':2,'other_M_points':5}},f,ensure_ascii=False,indent=2)
display(monthly_export_audit)
print(f'Đã lưu toàn bộ khách-tháng tại: {ALL_MONTHS_DIR}')
# Đọc lại khi cần (bảng lớn ~13.65 triệu dòng):
# paths=sorted(ALL_MONTHS_DIR.glob('rfm_*.parquet'))
# all_rfm=pd.concat([pd.read_parquet(p) for p in paths],ignore_index=True)
